In [1]:
import pandas as pd

In [2]:
# ============================================================
# HOME DEPOT CORPORATE BANKING CASE
# FY2025 / February 1, 2026
# $ billions unless otherwise stated
# ============================================================

hd = {
    "revenue": 164.683,
    "operating_income": 20.890,
    "depreciation_amortization": 3.273,
    "cash": 1.389,

    "short_term_debt": 4.464,
    "current_long_term_debt": 4.967,
    "long_term_debt": 46.341,
    "finance_leases": 2.963,

    "interest_expense": 2.412,
    "operating_cash_flow": 16.325,
    "capex": 3.679,

    "commercial_paper_capacity": 11.0,
    "commercial_paper_outstanding": 4.464,
    "backup_credit_facilities": 11.0
}

hd["ebitda"] = (
    hd["operating_income"]
    + hd["depreciation_amortization"]
)

# Funded debt excluding finance leases
hd["funded_debt"] = (
    hd["short_term_debt"]
    + hd["current_long_term_debt"]
    + hd["long_term_debt"]
    - hd["finance_leases"]
)

hd["net_debt"] = (
    hd["funded_debt"]
    - hd["cash"]
)

hd["free_cash_flow"] = (
    hd["operating_cash_flow"]
    - hd["capex"]
)

hd["debt_to_ebitda"] = (
    hd["funded_debt"]
    / hd["ebitda"]
)

hd["interest_coverage"] = (
    hd["operating_income"]
    / hd["interest_expense"]
)

hd

{'revenue': 164.683,
 'operating_income': 20.89,
 'depreciation_amortization': 3.273,
 'cash': 1.389,
 'short_term_debt': 4.464,
 'current_long_term_debt': 4.967,
 'long_term_debt': 46.341,
 'finance_leases': 2.963,
 'interest_expense': 2.412,
 'operating_cash_flow': 16.325,
 'capex': 3.679,
 'commercial_paper_capacity': 11.0,
 'commercial_paper_outstanding': 4.464,
 'backup_credit_facilities': 11.0,
 'ebitda': 24.163,
 'funded_debt': 52.809000000000005,
 'net_debt': 51.42,
 'free_cash_flow': 12.645999999999999,
 'debt_to_ebitda': 2.1855315978976124,
 'interest_coverage': 8.660862354892206}

In [3]:
# ============================================================
# LIQUIDITY
# ============================================================

unused_cp_capacity = (
    hd["commercial_paper_capacity"]
    - hd["commercial_paper_outstanding"]
)

liquidity = pd.DataFrame({
    "Liquidity Source": [
        "Cash & Cash Equivalents",
        "Unused Commercial Paper Capacity",
        "Backup Credit Facilities"
    ],
    "Amount ($B)": [
        hd["cash"],
        unused_cp_capacity,
        hd["backup_credit_facilities"]
    ]
})

liquidity

,Liquidity Source,Amount ($B)
0,Cash & Cash Equivalents,1.389
1,Unused Commercial Paper Capacity,6.536
2,Backup Credit Facilities,11.000


In [4]:
# ============================================================
# LONG-TERM DEBT MATURITY PROFILE
# ============================================================

debt_maturities = pd.DataFrame({
    "Fiscal Year": [
        "2026",
        "2027",
        "2028",
        "2029",
        "2030",
        "Thereafter"
    ],
    "Principal ($B)": [
        4.684,
        3.625,
        3.115,
        3.852,
        2.072,
        32.049
    ]
})

debt_maturities

,Fiscal Year,Principal ($B)
0,2026,4.684
1,2027,3.625
2,2028,3.115
3,2029,3.852
4,2030,2.072
5,Thereafter,32.049


In [5]:
# ============================================================
# PROPOSED $3B FINANCING
# ============================================================

new_debt = 3.0
new_debt_rate = 0.06

incremental_interest = (
    new_debt
    * new_debt_rate
)

pro_forma_debt = (
    hd["funded_debt"]
    + new_debt
)

pro_forma_interest = (
    hd["interest_expense"]
    + incremental_interest
)

pro_forma_debt_to_ebitda = (
    pro_forma_debt
    / hd["ebitda"]
)

pro_forma_interest_coverage = (
    hd["operating_income"]
    / pro_forma_interest
)

pro_forma_fcf = (
    hd["free_cash_flow"]
    - incremental_interest
)

financing_summary = pd.DataFrame({
    "Metric": [
        "Funded Debt",
        "Debt / EBITDA",
        "Interest Expense",
        "Interest Coverage",
        "Free Cash Flow"
    ],
    "Current": [
        hd["funded_debt"],
        hd["debt_to_ebitda"],
        hd["interest_expense"],
        hd["interest_coverage"],
        hd["free_cash_flow"]
    ],
    "Pro Forma + $3B": [
        pro_forma_debt,
        pro_forma_debt_to_ebitda,
        pro_forma_interest,
        pro_forma_interest_coverage,
        pro_forma_fcf
    ]
})

financing_summary

,Metric,Current,Pro Forma + $3B
0,Funded Debt,52.809000,55.809000
1,Debt / EBITDA,2.185532,2.309688
2,Interest Expense,2.412000,2.592000
3,Interest Coverage,8.660862,8.059414
4,Free Cash Flow,12.646000,12.466000


In [6]:
# ============================================================
# FINANCING STRESS TEST
# ============================================================

base_operating_margin = (
    hd["operating_income"]
    / hd["revenue"]
)

base_ebitda_margin = (
    hd["ebitda"]
    / hd["revenue"]
)

scenarios = [
    {
        "Scenario": "Base",
        "Revenue Shock": 0.00,
        "Margin Shock": 0.00,
        "Interest Shock": 0.00
    },
    {
        "Scenario": "Moderate Downside",
        "Revenue Shock": -0.05,
        "Margin Shock": -0.01,
        "Interest Shock": 0.10
    },
    {
        "Scenario": "Severe Downside",
        "Revenue Shock": -0.10,
        "Margin Shock": -0.02,
        "Interest Shock": 0.20
    }
]

results = []

for s in scenarios:

    stressed_revenue = (
        hd["revenue"]
        * (1 + s["Revenue Shock"])
    )

    stressed_ebitda_margin = (
        base_ebitda_margin
        + s["Margin Shock"]
    )

    stressed_operating_margin = (
        base_operating_margin
        + s["Margin Shock"]
    )

    stressed_ebitda = (
        stressed_revenue
        * stressed_ebitda_margin
    )

    stressed_ebit = (
        stressed_revenue
        * stressed_operating_margin
    )

    stressed_interest = (
        hd["interest_expense"]
        * (1 + s["Interest Shock"])
        + incremental_interest
    )

    debt_to_ebitda = (
        pro_forma_debt
        / stressed_ebitda
    )

    coverage = (
        stressed_ebit
        / stressed_interest
    )

    ocf_conversion = (
        hd["operating_cash_flow"]
        / hd["ebitda"]
    )

    stressed_ocf = (
        stressed_ebitda
        * ocf_conversion
    )

    stressed_fcf = (
        stressed_ocf
        - hd["capex"]
        - incremental_interest
    )

    results.append({
        "Scenario": s["Scenario"],
        "Revenue ($B)": stressed_revenue,
        "EBITDA ($B)": stressed_ebitda,
        "Debt / EBITDA": debt_to_ebitda,
        "Interest Coverage": coverage,
        "FCF ($B)": stressed_fcf
    })

scenario_df = pd.DataFrame(results)

scenario_df.round(2)

,Scenario,Revenue ($B),EBITDA ($B),Debt / EBITDA,Interest Coverage,FCF ($B)
0,Base,164.68,24.16,2.31,8.06,12.47
1,Moderate Downside,156.45,21.39,2.61,6.45,10.59
2,Severe Downside,148.21,18.78,2.97,5.15,8.83


In [7]:
# ============================================================
# DEBT CAPACITY
# ============================================================

base_max_leverage = 2.50
severe_max_leverage = 3.00
minimum_coverage = 5.00

severe_row = scenario_df[
    scenario_df["Scenario"] == "Severe Downside"
].iloc[0]

severe_ebitda = severe_row["EBITDA ($B)"]

severe_revenue = (
    hd["revenue"] * 0.90
)

severe_ebit = (
    severe_revenue
    * (base_operating_margin - 0.02)
)

base_leverage_capacity = (
    base_max_leverage
    * hd["ebitda"]
    - hd["funded_debt"]
)

severe_leverage_capacity = (
    severe_max_leverage
    * severe_ebitda
    - hd["funded_debt"]
)

severe_interest_capacity = (
    (
        severe_ebit / minimum_coverage
        - hd["interest_expense"] * 1.20
    )
    / new_debt_rate
)

capacity = pd.DataFrame({
    "Constraint": [
        "Base Leverage Capacity",
        "Severe Leverage Capacity",
        "Severe Coverage Capacity"
    ],
    "Additional Debt Capacity ($B)": [
        base_leverage_capacity,
        severe_leverage_capacity,
        severe_interest_capacity
    ]
})

capacity.round(2)

,Constraint,Additional Debt Capacity ($B)
0,Base Leverage Capacity,7.60
1,Severe Leverage Capacity,3.54
2,Severe Coverage Capacity,4.55
